<a href="https://colab.research.google.com/github/debojit11070/deep-learning/blob/main/language_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModel
import torch
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
import re
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Download NLTK data
nltk.download('punkt', quiet=True)

# Read the CSV file
def load_data(file_path):
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Loaded {len(df)} samples from {file_path}")
        return df
    except Exception as e:
        logging.error(f"Error loading CSV: {e}")
        return None

# Preprocess text: clean and normalize
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize spaces
    return text

# Extract features using TF-IDF (traditional ML approach)
def extract_tfidf_features(texts, max_features=5000):
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words=None)
    features = vectorizer.fit_transform(texts)
    return features, vectorizer

# Extract embeddings using a pre-trained LLM (e.g., XLM-RoBERTa for multilingual support)
def extract_llm_embeddings(texts, model_name="xlm-roberta-base", batch_size=8):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
            outputs = model(**inputs)
            # Use [CLS] token embedding
            batch_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
            embeddings.extend(batch_embeddings)
            logging.info(f"Processed batch {i//batch_size + 1}/{len(texts)//batch_size + 1}")

    return np.array(embeddings)

# Train and evaluate a classifier
def train_and_evaluate(X, y, model_name="Model"):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    logging.info(f"\nClassification Report for {model_name}:\n")
    print(classification_report(y_test, y_pred))
    return classifier

# Main function
def main():
    # Load data
    file_path = "language.csv"
    df = load_data(file_path)
    if df is None:
        return

    # Preprocess texts
    df['Text'] = df['Text'].apply(preprocess_text)
    texts = df['Text'].tolist()
    labels = df['language'].tolist()

    # Log language distribution
    lang_counts = Counter(labels)
    logging.info("Language distribution:")
    for lang, count in lang_counts.items():
        logging.info(f"{lang}: {count}")

    # Traditional ML: TF-IDF + RandomForest
    logging.info("Training TF-IDF + RandomForest model...")
    X_tfidf, vectorizer = extract_tfidf_features(texts)
    train_and_evaluate(X_tfidf, labels, "TF-IDF + RandomForest")

    # LLM-enhanced: XLM-RoBERTa embeddings + RandomForest
    logging.info("Extracting LLM embeddings...")
    X_llm = extract_llm_embeddings(texts, model_name="xlm-roberta-base")
    train_and_evaluate(X_llm, labels, "XLM-RoBERTa + RandomForest")

    # Simulate multi-LLM approach: Combine XLM-RoBERTa and BERT embeddings
    logging.info("Simulating multi-LLM approach...")
    X_bert = extract_llm_embeddings(texts, model_name="bert-base-multilingual-cased")
    X_combined = np.concatenate([X_llm, X_bert], axis=1)
    train_and_evaluate(X_combined, labels, "Combined LLMs + RandomForest")

if __name__ == "__main__":
    main()

              precision    recall  f1-score   support

      Arabic       1.00      1.00      1.00       202
     Chinese       0.50      0.03      0.07       201
       Dutch       1.00      0.99      0.99       230
     English       0.83      0.98      0.90       194
    Estonian       0.97      0.95      0.96       200
      French       0.97      0.98      0.98       188
       Hindi       1.00      0.98      0.99       208
  Indonesian       1.00      0.99      0.99       213
    Japanese       0.39      0.96      0.55       194
      Korean       0.99      0.96      0.97       190
       Latin       0.97      0.93      0.95       210
     Persian       1.00      0.99      0.99       196
   Portugese       0.98      0.99      0.99       194
      Pushto       1.00      0.96      0.98       196
    Romanian       0.98      0.98      0.98       197
     Russian       1.00      0.95      0.97       213
     Spanish       0.99      0.97      0.98       199
     Swedish       0.99    

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]